# Trash

A {class}`~pylabrobot.resources.trash.Trash` represents a disposal area on a liquid-handler
deck. It inherits the dimensions and positioning behavior of
{class}`~pylabrobot.resources.container.Container`, while liquid-handling operations treat it
as a terminal destination: discarded tips and resources are removed from tracked use rather
than assigned as children of the trash.

## Creating a trash area

Create a `Trash` with the physical dimensions of the disposal opening. PyLabRobot's default
length unit is millimeters:

In [ ]:
from pylabrobot.resources import Coordinate, Trash

trash = Trash(
  name="trash",
  size_x=120.0,
  size_y=80.0,
  size_z=50.0,
)

assert trash.category == "trash"
assert trash.get_anchor(x="center", y="center", z="top") == Coordinate(60, 40, 50)

## Adding trash to a deck

Many vendor-specific decks include their trash area automatically. For a custom
{class}`~pylabrobot.resources.deck.Deck`, assign the resource at its measured deck location.
The standard trash must be named `trash`, because
{meth}`~pylabrobot.resources.Deck.get_trash_area` uses that name to find it:

In [ ]:
from pylabrobot.resources import Deck

deck = Deck(name="deck", size_x=600.0, size_y=400.0, size_z=100.0)
trash_location = Coordinate(470.0, 10.0, 0.0)
deck.assign_child_resource(trash, location=trash_location)

assert deck.get_trash_area() is trash
assert trash.get_location_wrt(deck) == trash_location

## Discarding tips

`LiquidHandler.discard_tips()` uses the deck's standard trash as the drop destination. The
following example uses the Chatterbox simulation backend, which prints commands without
connecting to hardware:

In [ ]:
from pylabrobot.legacy.liquid_handling import LiquidHandler
from pylabrobot.legacy.liquid_handling.backends import LiquidHandlerChatterboxBackend
from pylabrobot.resources import hamilton_96_tiprack_300uL_filter, set_tip_tracking

set_tip_tracking(enabled=True)
tip_rack = hamilton_96_tiprack_300uL_filter(name="tips")
deck.assign_child_resource(tip_rack, location=Coordinate(20.0, 150.0, 0.0))

lh = LiquidHandler(backend=LiquidHandlerChatterboxBackend(num_channels=8), deck=deck)
await lh.setup()
await lh.pick_up_tips([tip_rack.get_item("A1")])
await lh.discard_tips(use_channels=[0])

assert not lh.head[0].has_tip
assert trash.children == []
await lh.stop()

## Clearing a deck

{meth}`~pylabrobot.resources.Deck.clear` preserves trash by default while removing other
deck resources. Pass `include_trash=True` when the trash area should also be detached:

In [ ]:
from pylabrobot.resources import Resource

work_area = Resource(name="work_area", size_x=100.0, size_y=80.0, size_z=20.0)
deck.assign_child_resource(work_area, location=Coordinate(20.0, 20.0, 0.0))

deck.clear()
assert deck.has_resource("trash")
assert not deck.has_resource("work_area")

deck.clear(include_trash=True)
assert not deck.has_resource("trash")

## Other disposal operations

Use `LiquidHandler.drop_tips()` when you need to target a `Trash` explicitly. Decks with a
separate 96-head disposal area can expose it through
{meth}`~pylabrobot.resources.Deck.get_trash_area96` for `LiquidHandler.discard_tips96()`.

A trash area does not retain discarded objects as children or record how many tips it contains.
Use external waste-capacity tracking when a workflow must stop before the physical bin is full.